- to do: decidere quanti anni sono un periodo.
- in input il file di quegli anni.

ora test con file un_1948_OK.txt

uso una versione lite di fasttext perché non mi interessano tutte e 300 le dimensioni

In [ ]:
import fasttext
import multiprocessing
from pathlib import Path

In [ ]:
# qui da cambiare con un ciclo su tutti i file, per ora è un esmepio
input_file = Path("../year_clean/un_1948_OK.txt")  
model_out = Path("fasttext_un_1948_lite")   # prefisso modello output

In [ ]:
# TRAINING FASTTEXT LEGGERO


print("Avvio training FastText 1948...")

model = fasttext.train_unsupervised(
    input=str(input_file),
    model="skipgram",
    dim=100,
    minn=3,
    maxn=5,
    wordNgrams=1,
    minCount=5,
    epoch=5,
    bucket=2000000,
    thread=multiprocessing.cpu_count()
)

# salva modello binario pronto per embedding
model.save_model(f"{model_out}.bin")
print(f"Modello binario salvato come {model_out}.bin")

# per fare un testo ci ha messo 22 secondi


In [ ]:
# estrai vocabolario e vettori
vocab = model.get_words()
word_vectors = {w: model.get_word_vector(w) for w in vocab}

print(vocab)
print(word_vectors)

to do: a questo punto posso eliminare stopwords e fare lemmatizzazione
to do: capire come gestire lemmatizzazione. 

Qui sotto ho provato a vedere due parole per vedere se conviene lemmatizzare o no.
Da questo esperimento io lascerei i token e non i lemmi = perdiamo troppa informazione.

In [ ]:
# carica modello addestrato
model = fasttext.load_model("fasttext_un_1948_lite.bin")



# parola di interesse
parola = "ha"

# verifica se la parola è nel vocabolario
if parola in model.get_words():
    vettore = model.get_word_vector(parola)
    print(f"Vettore di '{parola}':")
    print(vettore)
    print(f"Lunghezza del vettore: {len(vettore)}")
else:
    print(f"La parola '{parola}' non è nel vocabolario del modello")


In [ ]:
# parola di interesse
parola = "avere"

# verifica se la parola è nel vocabolario
if parola in model.get_words():
    vettore = model.get_word_vector(parola)
    print(f"Vettore di '{parola}':")
    print(vettore)
    print(f"Lunghezza del vettore: {len(vettore)}")
else:
    print(f"La parola '{parola}' non è nel vocabolario del modello  ")


In [ ]:
import fasttext
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np



# parole da confrontare
parola1 = "ha"
parola2 = "avere"

# controlla che siano nel vocabolario
vocab = model.get_words()
if parola1 in vocab and parola2 in vocab:
    # ottieni vettori
    vec1 = model.get_word_vector(parola1)
    vec2 = model.get_word_vector(parola2)
    
    # calcola similarità coseno
    sim = cosine_similarity([vec1], [vec2])[0][0]
    distanza = 1 - sim  # distanza coseno
    
    print(f"Similarità tra '{parola1}' e '{parola2}': {sim:.4f}")
    print(f"Distanza coseno tra '{parola1}' e '{parola2}': {distanza:.4f}")
else:
    print("Una delle due parole non è nel vocabolario del modello")


In [ ]:
# dal vocab voglio eliminare stopwords
# e parole formate da <3 lettere
from nltk.corpus import stopwords

# carica stopwords italiane
import nltk
nltk.download('stopwords')

stopwords_it = set(stopwords.words('italian'))

# filtra vocab: elimina stopwords e parole <3 lett
vocab_filtrato = [w for w in vocab if w not in stopwords_it and len(w) > 3]

print(f"Vocabolario originale: {len(vocab)} parole")
print(f"Vocabolario filtrato: {len(vocab_filtrato)} parole")   
